# 01 — Fetch Contacts

Pulls every active Dataverse contact whose parent account has
`accountcategorycode = 1`, a non-null `accountnumber`, and `vgr_datasource =
'vBill'`. Uses keyset pagination on `contactid`.

**Output**: `migration_data/01_contacts_by_account.csv`, one row per
contact, keyed by the *original* vBill `AccountCode` (not the two bucket
accounts every subscription actually lands on in this migration).

> **Output**: `migration_data/01_contacts_raw.csv` — every contact, one row
> each, used by `04_Create_Accounts.ipynb` to build the full `contact[]`
> block on each account (multiple contacts per account, same shape as the
> original per-customer migration notebook). `migration_data/01_contacts_by_account.csv`
> is a smaller one-row-per-account summary (primary contact only), used by
> `07_Create_Subscription_Orders.ipynb` as an optional order-level
> enrichment for subscriptions that land on the two shared "bucket"
> accounts rather than a proper per-customer account — see
> `ATTACH_CONTACT_SUMMARY_TO_ORDER` in `onebill_common.py`.

## 1. Setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403

logger = get_logger("fetch_contacts")


python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 27
python-dotenv could not parse statement starting at line 32
python-dotenv could not parse statement starting at line 38
python-dotenv could not parse statement starting at line 44


## 2. Fetch from Dataverse

In [2]:
dataverse_token = get_dataverse_token()
df_contacts = get_contacts(dataverse_token)
logger.info(f"Fetched {len(df_contacts):,} contacts from Dataverse")
df_contacts.head()


Contacts page 1: fetched 5,000 (total so far: 5,000)
Contacts page 2: fetched 5,000 (total so far: 10,000)
Contacts page 3: fetched 5,000 (total so far: 15,000)
Contacts page 4: fetched 5,000 (total so far: 20,000)
Contacts page 5: fetched 5,000 (total so far: 25,000)
Contacts page 6: fetched 5,000 (total so far: 30,000)
Contacts page 7: fetched 5,000 (total so far: 35,000)
Contacts page 8: fetched 5,000 (total so far: 40,000)


2026-07-31 10:42:20,345 [INFO] Fetched 41,893 contacts from Dataverse


Contacts page 9: fetched 1,893 (total so far: 41,893)


,@odata.etag,mobilephone,contactid,lastname,vgr_contactcode,firstname,fullname,emailaddress1,AccountCode.accountnumber,telephone1,vgr_contacttypes
0,"W/""4895868026""",021569156,7266d549-cbfa-ee11-9f89-000d3a6a0933,Prentice,C-00090123,Lynn,Lynn Prentice,lynn.prentice@gmail.com,47621153,NaN,NaN
1,"W/""4407698861""",0212141547,3b4848b4-7c5b-f011-bec1-000d3a6a2a4e,Sibbe,C-00100775,Judy,Judy Sibbe,sibbe@actrix.co.nz,99993083,NaN,NaN
2,"W/""4632705577""",N/A,83b8ffe3-c69b-ef11-8a69-000d3a6a332a,Rose,C-00094944,Angel,Angel Rose,accounts@techspanonline.com,99999381,6498276567,NaN
3,"W/""2990831880""",0210464741,0abb6f81-7d3d-ef11-a316-000d3a6a3623,Matthews,C-00092237,Angela,Angela Matthews,accounts@agedadvisor.co.nz,99937383,NaN,NaN
4,"W/""5404627661""",NaN,7b967bab-6c69-ef11-a670-000d3a6a3b2b,Clarke,C-00093561,Andre,Andre Clarke,andre.clarke@hbtech.co.nz,99993096,NaN,287790008


## 3. Clean + rename columns, apply defaults for blanks

In [3]:
df_contacts = df_contacts.drop(
    columns=[c for c in ["@odata.etag", "fullname"] if c in df_contacts.columns],
    errors="ignore",
)

df_contacts = df_contacts.rename(columns={
    "vgr_contactcode":         "ContactCode",
    "vgr_contacttypes":        "Dynamics_ContactTypes",
    "telephone1":              "PhoneWork",
    "mobilephone":             "PhoneMobile",
    "emailaddress1":           "EmailAddresses",
    "firstname":               "FirstName",
    "lastname":                "LastName",
    LINKED_ACCOUNTNUMBER_COL:  "AccountCode",
})

df_contacts["FirstName"]      = df_contacts["FirstName"].replace("", pd.NA).fillna(DEFAULT_FIRST_NAME)
df_contacts["LastName"]       = df_contacts["LastName"].replace("", pd.NA).fillna(DEFAULT_LAST_NAME)
df_contacts["EmailAddresses"] = df_contacts["EmailAddresses"].replace("", pd.NA).fillna(DEFAULT_EMAIL)

df_contacts["BillingContact"] = df_contacts["ContactCode"].str.startswith("BILLING")
df_contacts["OneBill_ContactType"] = df_contacts["ContactCode"].apply(
    lambda x: "0" if str(x).startswith("BILLING") else "1"
)

# Expand the Dynamics contact-type codes into their OneBill labels, as a
# pipe-separated string (kept flat for the CSV hand-off).
df_contacts["ContactTypeLabels"] = df_contacts["Dynamics_ContactTypes"].apply(
    lambda raw: "|".join(parse_contact_types(raw))
)

logger.info(f"Contacts after default substitution: {len(df_contacts):,}")
df_contacts.head()


2026-07-31 10:42:20,801 [INFO] Contacts after default substitution: 41,893


,PhoneMobile,contactid,LastName,ContactCode,FirstName,EmailAddresses,AccountCode,PhoneWork,Dynamics_ContactTypes,BillingContact,OneBill_ContactType,ContactTypeLabels
0,021569156,7266d549-cbfa-ee11-9f89-000d3a6a0933,Prentice,C-00090123,Lynn,lynn.prentice@gmail.com,47621153,NaN,NaN,False,1,
1,0212141547,3b4848b4-7c5b-f011-bec1-000d3a6a2a4e,Sibbe,C-00100775,Judy,sibbe@actrix.co.nz,99993083,NaN,NaN,False,1,
2,N/A,83b8ffe3-c69b-ef11-8a69-000d3a6a332a,Rose,C-00094944,Angel,accounts@techspanonline.com,99999381,6498276567,NaN,False,1,
3,0210464741,0abb6f81-7d3d-ef11-a316-000d3a6a3623,Matthews,C-00092237,Angela,accounts@agedadvisor.co.nz,99937383,NaN,NaN,False,1,
4,NaN,7b967bab-6c69-ef11-a670-000d3a6a3b2b,Clarke,C-00093561,Andre,andre.clarke@hbtech.co.nz,99993096,NaN,287790008,False,1,Communication


## 4. Primary-contact-per-account summary

One row per `AccountCode` — the billing contact if there is one, otherwise
the first contact seen. This is the row `07_Create_Subscription_Orders.ipynb`
looks up when attaching a contact summary to an order.

In [4]:
contacts_by_account = index_contacts_by_account(df_contacts)
logger.info(
    f"Indexed {sum(len(v) for v in contacts_by_account.values()):,} contacts "
    f"across {len(contacts_by_account):,} accounts"
)

primary_rows = []
for account_code, contacts in contacts_by_account.items():
    primary = contacts[0]  # BILLING- contact first, per index_contacts_by_account
    primary_rows.append({
        "AccountCode":  account_code,
        "ContactName":  f"{primary.get('FirstName', '')} {primary.get('LastName', '')}".strip(),
        "ContactEmail": primary.get("EmailAddresses"),
        "ContactPhone": primary.get("PhoneMobile") or primary.get("PhoneWork"),
        "ContactCount": len(contacts),
    })

df_primary_contacts = pd.DataFrame(primary_rows)
df_primary_contacts.head()


2026-07-31 10:42:26,389 [INFO] Indexed 41,885 contacts across 37,552 accounts


,AccountCode,ContactName,ContactEmail,ContactPhone,ContactCount
0,47621153,Lyn Prentice,home@primary.geek.nz,+6421569156,3
1,99993083,John Judy Sibbe,sibbe@actrix.gen.nz,NaN,2
2,99999381,John Techspan Group,accounts@techspanonline.com,+64275701703,6
3,99937383,Angela Matthews,accounts@agedadvisor.co.nz,+64211387064,4
4,99993096,John Hawkes Bay Technologies Limited,Accounts@hbtech.co.nz,NaN,14


## 5. Save

In [5]:
save_df("contacts", df_primary_contacts)
save_df("contacts_raw", df_contacts)  # every contact, not just one-per-account — used by 04_Create_Accounts.ipynb


Saved 37,552 rows -> migration_data\01_contacts_by_account.csv
Saved 41,893 rows -> migration_data\01_contacts_raw.csv
